# Probability calibration of classifiers: scikit-learn vs Moxie

**Reference example:** `../scikit-learn/examples/calibration/plot_calibration.py` at scikit-learn
`4358542a1bd39af2e7e1cbace8b9fed067473c5b`
([gallery](https://scikit-learn.org/dev/auto_examples/calibration/plot_calibration.html)).

**Scenario:** 50 000 three-blob samples relabelled into two classes, random sample weights,
a 10 % training split. A `GaussianNB` is compared uncalibrated, with
`CalibratedClassifierCV(cv=2, method="isotonic")` and with `method="sigmoid"`, both fitted with
`sample_weight`, and scored with the weighted Brier loss.

**Adaptations:**
- Moxie's `CalibratedClassifierCV` takes a `CalibrationEstimator` adapter (documented in
  `docs/calibration/family.md`); the `%%mojo` cell wraps Moxie's production `GaussianNB` in a
  ~60-line adapter that converts the Float64 targets to `List[Int]`, forwards sample weights
  (sklearn also forwards them because `GaussianNB.fit` accepts `sample_weight`) and returns
  `predict_proba` as the calibration response (GaussianNB has no `decision_function`).
- The data, split and weights are generated by the official Python code and shared as `.npy`
  files; Brier losses are computed in Python from each side's probabilities.
- The extra cells after the official code compare fitted attributes (`classes_`, per-fold
  sigmoid `a_`/`b_`, isotonic thresholds), `ensemble=False`, `method="temperature"`, a
  `FrozenEstimator` prefit calibration, the calibration curve and `CalibrationDisplay` statistics.

**Determinism:** fully deterministic (fixed `random_state`, `StratifiedKFold` without shuffling,
deterministic GaussianNB). Isotonic and temperature calibration and every curve are compared
numerically; sigmoid coefficients differ only by optimizer (sklearn L-BFGS-B with `gtol=1e-6`,
Moxie Newton with backtracking), see the tolerance notes.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "notebooks/_support/moxie_nb.py").exists())
sys.path.insert(0, str(ROOT / "notebooks/_support"))
%load_ext moxie_nb
nb = moxie_nb.Session("calibration/plot_calibration")

## 1. scikit-learn (official example code)

In [2]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split

n_samples = 50000
n_bins = 3  # use 3 bins for calibration_curve as we have 3 clusters here

# Generate 3 blobs with 2 classes where the second blob contains
# half positive samples and half negative samples. Probability in this
# blob is therefore 0.5.
centers = [(-5, -5), (0, 0), (5, 5)]
X, y = make_blobs(n_samples=n_samples, centers=centers, shuffle=False, random_state=42)

y[: n_samples // 2] = 0
y[n_samples // 2 :] = 1
sample_weight = np.random.RandomState(42).rand(y.shape[0])

# split train, test for calibration
X_train, X_test, y_train, y_test, sw_train, sw_test = train_test_split(
    X, y, sample_weight, test_size=0.9, random_state=42
)

from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss
from sklearn.naive_bayes import GaussianNB

# With no calibration
clf = GaussianNB()
clf.fit(X_train, y_train)  # GaussianNB itself does not support sample-weights
prob_pos_clf = clf.predict_proba(X_test)[:, 1]

# With isotonic calibration
clf_isotonic = CalibratedClassifierCV(clf, cv=2, method="isotonic")
clf_isotonic.fit(X_train, y_train, sample_weight=sw_train)
prob_pos_isotonic = clf_isotonic.predict_proba(X_test)[:, 1]

# With sigmoid calibration
clf_sigmoid = CalibratedClassifierCV(clf, cv=2, method="sigmoid")
clf_sigmoid.fit(X_train, y_train, sample_weight=sw_train)
prob_pos_sigmoid = clf_sigmoid.predict_proba(X_test)[:, 1]

print("Brier score losses: (the smaller the better)")
clf_score = brier_score_loss(y_test, prob_pos_clf, sample_weight=sw_test)
print("No calibration: %1.3f" % clf_score)
clf_isotonic_score = brier_score_loss(y_test, prob_pos_isotonic, sample_weight=sw_test)
print("With isotonic calibration: %1.3f" % clf_isotonic_score)
clf_sigmoid_score = brier_score_loss(y_test, prob_pos_sigmoid, sample_weight=sw_test)
print("With sigmoid calibration: %1.3f" % clf_sigmoid_score)

Brier score losses: (the smaller the better)
No calibration: 0.104
With isotonic calibration: 0.084
With sigmoid calibration: 0.109


Additional reference computations for attribute-level parity (same estimators, same data).

In [3]:
from sklearn.calibration import calibration_curve, CalibrationDisplay
from sklearn.frozen import FrozenEstimator
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sk = {}
sk["iso_proba"] = clf_isotonic.predict_proba(X_test)
sk["sig_proba"] = clf_sigmoid.predict_proba(X_test)
sk["iso_pred"] = clf_isotonic.predict(X_test)
sk["sig_ab"] = np.array([[c.calibrators[0].a_, c.calibrators[0].b_] for c in clf_sigmoid.calibrated_classifiers_])
for k, c in enumerate(clf_isotonic.calibrated_classifiers_):
    sk[f"iso_X_thr{k}"] = c.calibrators[0].X_thresholds_
    sk[f"iso_y_thr{k}"] = c.calibrators[0].y_thresholds_
pooled = CalibratedClassifierCV(GaussianNB(), cv=2, method="isotonic", ensemble=False).fit(
    X_train, y_train, sample_weight=sw_train)
sk["pooled_proba"] = pooled.predict_proba(X_test)
temperature = CalibratedClassifierCV(GaussianNB(), cv=2, method="temperature").fit(
    X_train, y_train, sample_weight=sw_train)
sk["temp_proba"] = temperature.predict_proba(X_test)
sk["temp_beta"] = np.array([c.calibrators[0].beta_ for c in temperature.calibrated_classifiers_])
# Prefit: calibrate the already-fitted clf on the held-out half of the test split.
X_cal, y_cal, sw_cal = X_test[:20000], y_test[:20000], sw_test[:20000]
frozen = CalibratedClassifierCV(FrozenEstimator(clf), method="sigmoid").fit(X_cal, y_cal, sample_weight=sw_cal)
sk["frozen_proba"] = frozen.predict_proba(X_test[20000:])
sk["frozen_count"] = np.array([len(frozen.calibrated_classifiers_)])
for strategy in ["uniform", "quantile"]:
    t, p = calibration_curve(y_test, prob_pos_isotonic, n_bins=n_bins, strategy=strategy)
    sk[f"curve_{strategy}"] = np.vstack([t, p])
disp = CalibrationDisplay.from_estimator(clf_sigmoid, X_test, y_test, n_bins=10)
sk["disp_true"], sk["disp_pred"] = disp.prob_true, disp.prob_pred
print("sklearn display labels:", disp.ax_.get_xlabel(), "|", disp.line_.get_label())
plt.close(disp.figure_)

nb.share(X_train=X_train, y_train=y_train.astype(np.float64), sw_train=sw_train,
         X_test=X_test, y_test=y_test.astype(np.float64), sw_cal=sw_cal)

sklearn display labels: Mean predicted probability (Positive class: 1) | CalibratedClassifierCV


/home/strgenix/postdoc/projects/mojods/porting-sklearn/moxie/.pixi/envs/notebooks/lib/python3.14/site-packages/sklearn/calibration.py:396: UserWarning: Since FrozenEstimator does not appear to accept sample_weight, sample weights will only be used for the calibration itself. This can be caused by a limitation of the current scikit-learn API. See the following issue for more details: https://github.com/scikit-learn/scikit-learn/issues/21134. Be warned that the result of the calibration is likely to be incorrect.
  warnings.warn(


## 2. Moxie (native Mojo)

The same scenario through Moxie's public API: `GaussianNB` from `moxie.naive_bayes`, calibrated by
`moxie.calibration.CalibratedClassifierCV`. Learning and prediction happen in Mojo.

In [4]:
%%mojo plot_calibration
import numojo as nm
from moxie.calibration import (
    CalibrationEstimator, CalibrationDisplayEstimator, CalibratedClassifierCV,
)
from moxie.naive_bayes import GaussianNB, GaussianNBParams

comptime Array = nm.NDArray[DType.float64]


struct GNBAdapter(CalibrationDisplayEstimator, CalibrationEstimator):
    """Calibration adapter (docs/calibration/family.md) for Moxie's GaussianNB.

    Float64 targets become List[Int]; weights are forwarded like sklearn does;
    GaussianNB has no decision_function, so the response is predict_proba.
    """
    comptime Params = GaussianNBParams
    var model: GaussianNB[]

    def __init__(out self) raises:
        self.model = GaussianNB()

    def get_params(self, deep: Bool = True) raises -> Self.Params:
        return self.model.get_params(deep)

    def set_params[origin: MutOrigin, params_origin: ImmOrigin](
        ref[origin] self, ref[params_origin] params: Self.Params
    ) raises -> ref[self] Self:
        _ = self.model.set_params(params)
        return self

    def __sklearn_clone__(self) raises -> Self:
        var result = Self()
        result.model = self.model.__sklearn_clone__()
        return result^

    def __sklearn_is_fitted__(self) -> Bool:
        return self.model.__sklearn_is_fitted__()

    def fit(mut self, X: Array, y: Array, sample_weight: Optional[Array] = None) raises:
        var labels = List[Int]()
        for i in range(y.size):
            labels.append(Int(y.item(i)))
        var weights: Optional[List[Float64]] = None
        if sample_weight:
            var w = List[Float64]()
            for i in range(sample_weight.value().size):
                w.append(sample_weight.value().item(i))
            weights = w^
        _ = self.model.fit(X, labels, sample_weight=weights)

    def class_labels(self) raises -> Array:
        var result = Array(nm.Shape(len(self.model.classes_)))
        for i in range(len(self.model.classes_)):
            result[nm.Item(i)] = Float64(self.model.classes_[i])
        return result^

    def uses_probabilities(self) -> Bool:
        return True

    def response(self, X: Array) raises -> Array:
        return self.model.predict_proba(X)

    def calibration_probabilities(self, X: Array) raises -> Array:
        return self.model.predict_proba(X)

    def display_name(self) -> String:
        return "GaussianNB"

from moxie.calibration import FrozenCalibrationEstimator, calibration_curve, CalibrationDisplay
from moxie.frozen import FrozenEstimator
from moxie_nb_io import load, save, save_floats, save_strings


def positive(p: Array) raises -> Array:
    var out = Array(nm.Shape(p.shape[0]))
    for i in range(p.shape[0]):
        out[nm.Item(i)] = p.item(i, 1)
    return out^


def head(x: Array, n: Int, start: Int = 0) raises -> Array:
    var columns = x.shape[1] if x.ndim == 2 else 0
    var out = Array(nm.Shape(n, columns)) if columns > 0 else Array(nm.Shape(n))
    for i in range(n):
        if columns > 0:
            for j in range(columns):
                out[nm.Item(i, j)] = x.item(start + i, j)
        else:
            out[nm.Item(i)] = x.item(start + i)
    return out^


def main() raises:
    var X_train = load("X_train")
    var y_train = load("y_train")
    var sw_train = load("sw_train")
    var X_test = load("X_test")
    var y_test = load("y_test")

    # With no calibration (the example fits GaussianNB without weights)
    var clf = GNBAdapter()
    clf.fit(X_train, y_train)
    save("prob_pos_clf", positive(clf.calibration_probabilities(X_test)))

    # With isotonic calibration
    var clf_isotonic = CalibratedClassifierCV(clf.__sklearn_clone__(), cv=2, method="isotonic")
    _ = clf_isotonic.fit(X_train, y_train, sw_train.copy())
    var iso_proba = clf_isotonic.predict_proba(X_test)
    save("iso_proba", iso_proba)
    save("iso_pred", clf_isotonic.predict(X_test))
    save("classes", clf_isotonic.classes_)
    for k in range(len(clf_isotonic.calibrated_classifiers_)):
        ref pair = clf_isotonic.calibrated_classifiers_[k]
        save("iso_X_thr" + String(k), pair.isotonic[0].X_thresholds_)
        save("iso_y_thr" + String(k), pair.isotonic[0].y_thresholds_)

    # With sigmoid calibration
    var clf_sigmoid = CalibratedClassifierCV(clf.__sklearn_clone__(), cv=2, method="sigmoid")
    _ = clf_sigmoid.fit(X_train, y_train, sw_train.copy())
    save("sig_proba", clf_sigmoid.predict_proba(X_test))
    var ab = Array(nm.Shape(len(clf_sigmoid.calibrated_classifiers_), 2))
    for k in range(len(clf_sigmoid.calibrated_classifiers_)):
        ab[nm.Item(k, 0)] = clf_sigmoid.calibrated_classifiers_[k].sigmoid[0].a_
        ab[nm.Item(k, 1)] = clf_sigmoid.calibrated_classifiers_[k].sigmoid[0].b_
    save("sig_ab", ab)

    # ensemble=False pools out-of-fold responses and refits on all samples
    var pooled = CalibratedClassifierCV(GNBAdapter(), cv=2, method="isotonic", ensemble=False)
    _ = pooled.fit(X_train, y_train, sw_train.copy())
    save("pooled_proba", pooled.predict_proba(X_test))

    # Temperature scaling
    var temperature = CalibratedClassifierCV(GNBAdapter(), cv=2, method="temperature")
    _ = temperature.fit(X_train, y_train, sw_train.copy())
    save("temp_proba", temperature.predict_proba(X_test))
    var betas = List[Float64]()
    for pair in temperature.calibrated_classifiers_:
        betas.append(pair.temperature.beta_)
    save_floats("temp_beta", betas^)

    # Prefit through FrozenEstimator: ensemble="auto" becomes False
    var X_cal = head(X_test, 20000)
    var y_cal = head(y_test, 20000)
    var sw_cal = load("sw_cal")
    var frozen = CalibratedClassifierCV(
        FrozenCalibrationEstimator(FrozenEstimator(clf.copy())), method="sigmoid")
    _ = frozen.fit(X_cal, y_cal, sw_cal.copy())
    save("frozen_proba", frozen.predict_proba(head(X_test, X_test.shape[0] - 20000, 20000)))
    save_floats("frozen_count", [Float64(len(frozen.calibrated_classifiers_))])

    # Calibration curves and the display on the calibrated models
    var prob_pos_isotonic = positive(iso_proba)
    for strategy in ["uniform", "quantile"]:
        var curve = calibration_curve(y_test, prob_pos_isotonic, n_bins=3, strategy=strategy)
        var stacked = Array(nm.Shape(2, curve[0].size))
        for i in range(curve[0].size):
            stacked[nm.Item(0, i)] = curve[0].item(i)
            stacked[nm.Item(1, i)] = curve[1].item(i)
        save("curve_" + String(strategy), stacked)
    var display = CalibrationDisplay.from_estimator(clf_sigmoid, X_test, y_test, n_bins=10)
    save("disp_true", display.prob_true)
    save("disp_pred", display.prob_pred)
    print("moxie display labels:", display.ax_.value().xlabel, "|", display.line_.value().label.value())
    save_strings("labels", [display.ax_.value().xlabel, display.line_.value().label.value()])

moxie display labels: Mean predicted probability (Positive class: 1.0) | CalibratedClassifierCV


## 3. Side by side and parity

In [5]:
mx = nb.results["plot_calibration"]
from sklearn.metrics import brier_score_loss
scores = {
    "No calibration": (clf_score, brier_score_loss(y_test, mx["prob_pos_clf"], sample_weight=sw_test)),
    "Isotonic": (clf_isotonic_score, brier_score_loss(y_test, mx["iso_proba"][:, 1], sample_weight=sw_test)),
    "Sigmoid": (clf_sigmoid_score, brier_score_loss(y_test, mx["sig_proba"][:, 1], sample_weight=sw_test)),
}
import pandas as pd
moxie_nb.side_by_side(brier=pd.DataFrame(scores, index=["sklearn", "moxie"]).T,
                      sigmoid_ab=pd.DataFrame({"sklearn a_": sk["sig_ab"][:, 0], "moxie a_": mx["sig_ab"][:, 0],
                                               "sklearn b_": sk["sig_ab"][:, 1], "moxie b_": mx["sig_ab"][:, 1]}))

GNB = "GaussianNB variance reductions differ in summation order (naive_bayes family parity is ~1e-12)"
nb.compare("classes_", clf_isotonic.classes_.astype(float), mx["classes"], exact=True)
nb.compare("uncalibrated GaussianNB P(y=1)", prob_pos_clf, mx["prob_pos_clf"], rtol=1e-9, atol=1e-12, reason=GNB)
nb.compare("isotonic predict", sk["iso_pred"].astype(float), mx["iso_pred"], exact=True)
nb.compare("isotonic predict_proba", sk["iso_proba"], mx["iso_proba"], rtol=1e-9, atol=1e-11,
           reason="PAV is exact; inputs inherit the GaussianNB rounding above")
for k in range(2):
    nb.compare(f"isotonic fold {k} X_thresholds_", sk[f"iso_X_thr{k}"], mx[f"iso_X_thr{k}"], rtol=1e-9, atol=1e-12, reason=GNB)
    nb.compare(f"isotonic fold {k} y_thresholds_", sk[f"iso_y_thr{k}"], mx[f"iso_y_thr{k}"], rtol=1e-9, atol=1e-12,
               reason="weighted block means of identical blocks")
SIG = ("optimizer differs (sklearn L-BFGS-B stops at gtol=1e-6, Moxie Newton converges to 1e-8 gradient); "
       "family parity uses 5e-6 relative on probabilities")
nb.compare("sigmoid a_, b_ per fold", sk["sig_ab"], mx["sig_ab"], rtol=1e-4, atol=1e-6, reason=SIG)
nb.compare("sigmoid predict_proba", sk["sig_proba"], mx["sig_proba"], rtol=5e-6, atol=5e-6, reason=SIG)
nb.compare("ensemble=False isotonic predict_proba", sk["pooled_proba"], mx["pooled_proba"], rtol=1e-9, atol=1e-11,
           reason="exact PAV on out-of-fold GaussianNB probabilities")
TEMP = "bounded golden-section vs SciPy Brent on log-beta; family parity uses 5e-6 relative"
nb.compare("temperature beta_ per fold", sk["temp_beta"], mx["temp_beta"], rtol=1e-5, reason=TEMP)
nb.compare("temperature predict_proba", sk["temp_proba"], mx["temp_proba"], rtol=5e-6, atol=5e-6, reason=TEMP)
nb.compare("FrozenEstimator prefit: one calibrated pair", sk["frozen_count"].astype(float), mx["frozen_count"], exact=True)
nb.compare("FrozenEstimator prefit sigmoid predict_proba", sk["frozen_proba"], mx["frozen_proba"], rtol=5e-6, atol=5e-6, reason=SIG)
CURVE = "bin means of Float64 values; reduction order differs"
for strategy in ["uniform", "quantile"]:
    nb.compare(f"calibration_curve {strategy} (prob_true; prob_pred)", sk[f"curve_{strategy}"], mx[f"curve_{strategy}"],
               rtol=1e-9, atol=1e-11, reason=CURVE + "; inputs inherit isotonic tolerance")
nb.compare("CalibrationDisplay.from_estimator prob_true", sk["disp_true"], mx["disp_true"], rtol=1e-9, atol=1e-12,
           reason="bin occupancy identical; sigmoid probabilities differ < 5e-6 but no sample crosses a 0.1 edge")
nb.compare("CalibrationDisplay.from_estimator prob_pred", sk["disp_pred"], mx["disp_pred"], rtol=5e-6, atol=5e-6, reason=SIG)
nb.compare("display line label", np.array([disp.line_.get_label()]), np.array([mx["labels"][1]]), exact=True)
nb.note("display xlabel", "ADAPTED",
        f"sklearn {disp.ax_.get_xlabel()!r} vs moxie {mx['labels'][0]!r}: Moxie labels are Float64, so pos_label prints as 1.0 "
        "(sklearn prints its integer y labels); documented typed-label difference")
nb.note("default estimator=None (LinearSVC)", "GAP", "Moxie requires an explicit CalibrationEstimator adapter (docs/calibration/family.md)")

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), dpi=80, sharey=True)
order = np.lexsort((prob_pos_clf,))
for ax, title, (p0, p1, p2) in [
        (axes[0], "scikit-learn", (prob_pos_clf, prob_pos_isotonic, prob_pos_sigmoid)),
        (axes[1], "Moxie", (mx["prob_pos_clf"], mx["iso_proba"][:, 1], mx["sig_proba"][:, 1]))]:
    ax.plot(p0[order], "r", label="No calibration")
    ax.plot(p1[order], "g", linewidth=3, label="Isotonic calibration")
    ax.plot(p2[order], "b", linewidth=3, label="Sigmoid calibration")
    ax.plot(np.linspace(0, y_test.size, 51)[1::2], y_test[order].reshape(25, -1).mean(1), "k", linewidth=3, label="Empirical")
    ax.set_ylim([-0.05, 1.05]); ax.set_title(f"{title}: Gaussian naive Bayes probabilities")
    ax.set_xlabel("Instances sorted by uncalibrated GNB probability")
axes[0].set_ylabel("P(y=1)"); axes[0].legend(loc="upper left", fontsize=7)
plt.show()

/tmp/ipykernel_1084344/1011378366.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
nb.report()
nb.assert_parity()

check,status,sklearn shape,moxie shape,max |diff|,max rel,tolerance,detail,tolerance reason
classes_,PASS,"(2,)","(2,)",,,exact,0 mismatches,
uncalibrated GaussianNB P(y=1),PASS,"(45000,)","(45000,)",8.88e-16,1.99e-15,"rtol=1e-09, atol=1e-12",,GaussianNB variance reductions differ in summation order (naive_bayes family parity is ~1e-12)
isotonic predict,PASS,"(45000,)","(45000,)",,,exact,0 mismatches,
isotonic predict_proba,PASS,"(45000, 2)","(45000, 2)",6.62e-14,6.46e-13,"rtol=1e-09, atol=1e-11",,PAV is exact; inputs inherit the GaussianNB rounding above
isotonic fold 0 X_thresholds_,PASS,"(18,)","(18,)",1.22e-15,1.78e-14,"rtol=1e-09, atol=1e-12",,GaussianNB variance reductions differ in summation order (naive_bayes family parity is ~1e-12)
isotonic fold 0 y_thresholds_,PASS,"(18,)","(18,)",1.11e-16,2.38e-16,"rtol=1e-09, atol=1e-12",,weighted block means of identical blocks
isotonic fold 1 X_thresholds_,PASS,"(10,)","(10,)",2.11e-15,1.41e-14,"rtol=1e-09, atol=1e-12",,GaussianNB variance reductions differ in summation order (naive_bayes family parity is ~1e-12)
isotonic fold 1 y_thresholds_,PASS,"(10,)","(10,)",2.22e-16,4.46e-16,"rtol=1e-09, atol=1e-12",,weighted block means of identical blocks
"sigmoid a_, b_ per fold",PASS,"(2, 2)","(2, 2)",2.08e-08,4.32e-09,"rtol=0.0001, atol=1e-06",,"optimizer differs (sklearn L-BFGS-B stops at gtol=1e-6, Moxie Newton converges to 1e-8 gradient); family parity uses 5e-6 relative on probabilities"
sigmoid predict_proba,PASS,"(45000, 2)","(45000, 2)",5.67e-10,5.82e-09,"rtol=5e-06, atol=5e-06",,"optimizer differs (sklearn L-BFGS-B stops at gtol=1e-6, Moxie Newton converges to 1e-8 gradient); family parity uses 5e-6 relative on probabilities"
